### Problem Statement: 

This competition presents a chance to benchmark your sentiment-analysis ideas on the Rotten Tomatoes dataset. You are asked to label phrases on a scale of five values: negative, somewhat negative, neutral, somewhat positive, positive. Obstacles like sentence negation, sarcasm, terseness, language ambiguity, and many others make this task very challenging. Submissions are evaluated on classification accuracy (the percent of labels that are predicted correctly) for every parsed phrase.

### Solution Approach:

**Preprocessing Steps:** 
- Handling special char, extra white space
- Replacing the "!" with "exclamation" and 0-9 with "num" string.
- Handling `'s` and `'t`
- Removing stop words but keeping negative sentiment words like "no" "not" "never"
- Removing empty Phrases.

## Data Understanding

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import kagglehub
import matplotlib.pyplot as plt 
import seaborn as sns
import sklearn
import nltk
import re

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
sample = pd.read_csv("/kaggle/input/competitions/sentiment-analysis-on-movie-reviews/sampleSubmission.csv")
train = pd.read_csv("/kaggle/input/datasets/samratrm/movie-review/train.tsv", sep='\t')
test = pd.read_csv("/kaggle/input/datasets/samratrm/movie-review/test.tsv", sep='\t')
train.head()

In [ ]:
test.head()

In [ ]:
print(train.shape, test.shape)

The sentences are broken down into multiple phrases because when it gets built up piece by piece as you combine words into phrases, and each of those intermediate combinations can carry its own emotional tone that differs from the final sentence.

Take a sentence like "not a good movie." The word "good" alone is positive. The phrase "a good movie" is positive. But once you add "not," the whole sentence flips to negative.

In [ ]:
train[(train['SentenceId']==1) & (train['PhraseId']<10)]

**Example:**

"fans" word has a sentiment of Slightly Positive but another part of the same sentence is "would have a hard time sitting through this one" which has a negative sentiment.

In [ ]:
train[(train['SentenceId']==3) & (train['Sentiment'] == 3)]

In [ ]:
train[(train['SentenceId']==3) & (train['Sentiment'] == 0)]

In [ ]:
train.describe()

# Data Cleaning & EDA

### Missing and Duplicate Data 

In [ ]:
train.isna().sum() 

- There are no missing data and no duplicate rows in the training data. 

In [ ]:
 train['Phrase'].duplicated().sum()

In [ ]:
display(test.isna().sum())
display(test['Phrase'].duplicated().sum())

In [ ]:
display(train['Phrase'].str.lower().duplicated().sum())
display(test['Phrase'].str.lower().duplicated().sum())

Since the Duplicate phrase value is in test data. We can ignore it. 

There is one Phrase value missing in the test dataset. Let's fill it with a original sentence data since it must have the most context. 

In [ ]:
s_id = test[test['Phrase'].isna()]['SentenceId']
longest_lookup = (
    test.dropna(subset=['Phrase'])
        .assign(plen=lambda d: d['Phrase'].str.len())
        .sort_values('plen', ascending=False)
        .drop_duplicates('SentenceId')
        .set_index('SentenceId')['Phrase']
)

test.fillna( {'Phrase': test['SentenceId'].map(longest_lookup)},inplace=True)

In [ ]:
display(test.isna().sum())

### Noise or Invalid Data

In [ ]:
print((train['Phrase'].str.len() == 0).sum() , (test['Phrase'].str.len() == 0).sum())

- There is no empty Phrase string in train or test data.

In [ ]:
print((train['Phrase'].str.len() <= 3 ).sum())
print(list(train[train['Phrase'].str.len() <= 3 ]['Phrase'][:50]))

In [ ]:
print((test['Phrase'].str.len() <= 3 ).sum())
print(list(test[test['Phrase'].str.len() <= 3 ]['Phrase'][:50]))

- 666 train data and 462 in test data phrases are filled with stop words, numbers or symbols most of that will be filtered and then it will become an empty string.
- There are some meaningful words like "TV", "nod", "sad", "fun", "joy" etc

Let's analyse the length of the phrases before data cleaning step to compare the differnece

In [ ]:
train['char_len'] = train['Phrase'].str.len()
train['char_len'].hist(bins=30)

In [ ]:
print((((train['Phrase'].str.len() < 35).sum() / train.shape[0]) * 100) ,
      (((train['Phrase'].str.len() < 50).sum() / train.shape[0]) * 100) )

- ~60% of Phrase data has less than 35 character length
- ~72% of Phrase has less than 50 character length.
- It has a huge peak around the 10-20 character length.  

In [ ]:
sns.histplot(data=train,x='char_len',hue="Sentiment",stat='probability', multiple="stack",common_norm=False,alpha=0.5,bins=40)

- There is no major trend in the sentiment's divide in the histogram. All the sentiment follows a similar trend.

#### Phrase Cleaning:

In [ ]:
special_char_pattern = r'[^a-zA-Z0-9\s]'
exclamation_pattern = r'!+'
white_space_pattern = r'\s+'
empty_string = ""
space_string = " "
is_word_pattern = r"(\w+)'s\b"

is_word_replacement = r"\1 is"
# John's becomes "John is" but it's removed in stop word filter. So, it's a side effect that solves itself. 
not_word_pattern = r"(\w+)'t\b"
# not_word_replacement = r"\1 not"

# Matches words ending in 't (don't, dont) OR isolated shards (n't, nt, 't)
not_contract_pattern = r"\b[a-z]+'?t\b|'?\bn'?t\b|\b't\b"

num_pattern = r'\d+'
num_placeholder = " num "
contract_pattern = r"\b[a-z]+'?t\b|\bn'?t\b"
negative_stop_words = {
    'no', 'not', 'nor', 'neither', 'never', 'none', 'but',
    'ain', 'aren', "aren't", 'couldn', "couldn't", 'didn', "didn't", 
    'doesn', "doesn't", 'don', "don't", 'hadn', "hadn't", 'hasn', "hasn't", 
    'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 
    'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 
    'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"
}
emoji_map = {
    r":\)": " happy ",r":-\)": " happy ",r":D": " happy ",r":\(": " sad ",
    r":-\(": " sad ",r":'\(": " crying ",r";\)": " wink ",r":/": " unsure ",
    r":\|": " neutral ","😊": " happy ","😍": " love ","😢": " crying ",
    "😞": " sad ","😡": " angry ","👍": " good ","👎": " bad ",
}
NOT_CONTRACTIONS = {
    "won't": "will not",     "can't": "cannot",       "shan't": "shall not",
    "ain't": "is not",       "don't": "do not",       "doesn't": "does not",
    "didn't": "did not",     "isn't": "is not",       "aren't": "are not",
    "wasn't": "was not",     "weren't": "were not",   "haven't": "have not",
    "hasn't": "has not",     "hadn't": "had not",     "wouldn't": "would not",
    "couldn't": "could not", "shouldn't": "should not","mustn't": "must not",
    "needn't": "need not",   "mightn't": "might not",
}
not_contraction_pattern = re.compile(
    r"\b(" + "|".join(sorted(map(re.escape, NOT_CONTRACTIONS), key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

In [ ]:
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt')

bracket_artifacts = {'lrb', 'rrb'} 

stop_words = set(stopwords.words('english')) - negative_stop_words
stop_words.update(bracket_artifacts) # removing bracket 

In [ ]:
from nltk.tokenize import word_tokenize

def remove_stop_words(text): 
    words = word_tokenize(text)
    return [w for w in words if w.lower() not in stop_words]
  
def remove_extra_char_and_white_space(text):
    text = re.sub(white_space_pattern, space_string, text)
    text = re.sub(special_char_pattern, empty_string, text)
    return text.strip()

def replace_exclamation_and_num(text):
    def replace_exclamation(match):
        if len(match.group(0)) == 1:
            return ' exclamation '
        return ' exclamations '
    text = re.sub(exclamation_pattern, replace_exclamation, text)
    return re.sub(num_pattern, num_placeholder, text)

def expand_contraction(text):
    text = re.sub(is_word_pattern, is_word_replacement, text)
    def replace_match(match):
        word = match.group(0).lower()
        
        # Check exact match in dictionary (e.g., "don't")
        if word in NOT_CONTRACTIONS:
            return NOT_CONTRACTIONS[word]
            
        # Check dictionary without apostrophe (e.g., "dont")
        word_no_apostrophe = word.replace("'", "")
        if word_no_apostrophe in NOT_CONTRACTIONS:
            return NOT_CONTRACTIONS[word_no_apostrophe]
            
        # Fallback logic for isolated shards (e.g., "n't", "nt", "'t")
        if word in ["n't", "nt", "'t", "t"]:
            return "not"
        return word
    return re.sub(contract_pattern, replace_match, text)

**Note:**  sklearn's default tokenizer requires tokens of at least 2 characters, so single-digit numbers get silently dropped during vectorization regardless of what this function does.

In [ ]:
print(train['Phrase'].str.contains(r'\d').sum())

- Only ~2% of data has numbers

In [ ]:
def count_emojis(text):
    count = 0 
    for pattern, word in emoji_map.items():
        if bool(re.search(pattern, text)): 
            count += 1
    return count 

emoji_count = train['Phrase'].apply(count_emojis)
emoji_count.sum()

- There is no emoji or most used emoji in the Phrase data

In [ ]:
def clean_phrase(text): 
    text = replace_exclamation_and_num(text)
    text = text.lower()
    text = expand_contraction(text) 
    text = remove_extra_char_and_white_space(text)
    words = remove_stop_words(text)

    return space_string.join(words)

In [ ]:
train['clean_phrase'] = train['Phrase'].apply(clean_phrase)
test['clean_phrase'] = test['Phrase'].apply(clean_phrase)
train['clean_phrase'][:5]

In [ ]:
(train['clean_phrase'] == "").sum()

After cleaning there is 1068 empty rows or ~0.4% So, Lets drop it

In [ ]:
train = train[train['clean_phrase'] != "" ].reset_index(drop=True)
(train['clean_phrase'] == "").sum()

#### Phrase cleaning summary: 

- Preserves Critical Sentiment Noise: Instead of dropping expressive punctuation or numerical counts completely, it explicitly translates exclamation marks to "exclamation" or "exclamations" and standardizes digits to a generic "num" placeholder.

- Robust Contraction Expansion: Replaces standard possessive/verb shortcuts ('s to is) and maps comprehensive negative shortcuts (like won't or don't) to their full forms (will not, do not) via dictionary lookups and catches broken text shards (such as n't, nt, or standalone 't) produced by pre-tokenized inputs and intelligently forces them back to "not".

- Erases non-alphanumeric punctuation marks, strips out excess white spaces, and converts all content to lowercase to keep text distributions uniform.

- Context-Aware Stop Word Filtering: Cleans out standard grammatical filler words while explicitly shielding critical sentiment-reversing words (like no, not, never, and but) and treebank bracket artifacts (lrb, rrb) from being removed.

#### Character Length analysis

In [ ]:
train['char_len'] = train['clean_phrase'].str.len()
sns.histplot(data=train,x='char_len',hue="Sentiment",stat='probability', multiple="stack",common_norm=False,alpha=0.5,bins=40)

In [ ]:
train.boxplot(column="char_len",by='Sentiment')
plt.title("")

**Phrase Length by Sentiment Class**

The boxplot shows a clear **non-linear** relationship between phrase length and sentiment 
class. The length doesn't increase or decrease steadily from negative to positive, it dips 
sharply in the middle.

#### Most common words analysis

In [ ]:
from collections import Counter

def get_word_counts(reviews, n=20):
    words = []
    for text in reviews:
        words.extend(word_tokenize(text))
    return Counter(words).most_common(n)

print(f"Before Cleaning - Most common words in the dataset: {get_word_counts(train['Phrase'],30)}" )

In [ ]:
print(f"After cleanign - Most common words in the dataset: {get_word_counts(train['clean_phrase'],50)}" )

After cleaning almost all thw words makes sense and are meaningful to the classification. 

In [ ]:
all_words = []
for text in train['clean_phrase']:
    if isinstance(text, str):
        all_words.extend(word_tokenize(text))

top_words = [word for word,count in Counter(all_words).most_common(25)]

plot_data = []
for sentiment, group in train.groupby('Sentiment'):
    group_words = []
    for text in group['clean_phrase']:
        if isinstance(text,str):
            group_words.extend(word_tokenize(text))
    counts = Counter(group_words)

    for word in top_words:
        plot_data.append({
            "Word":word,
            "Count": counts[word],
            "Sentiment":sentiment
        })

df_plot = pd.DataFrame(plot_data)
df_plot.head()

In [ ]:
# Create the horizontal stacked bar plot
plt.figure(figsize=(10, 8))
sns.histplot(
    data=df_plot,
    y='Word',
    weights='Count',
    hue='Sentiment',
    multiple='stack',
    palette='coolwarm',  # Clean, distinct color gradient
    shrink=0.8
)

plt.title('Top Cleaned Words Count Segmented by Sentiment Class', fontsize=14)
plt.xlabel('Total Occurrences (Stacked by Sentiment)')
plt.ylabel('Words')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

**Top Words by Sentiment Class**

The top words are dominated by generic, class-agnostic terms — "film," "movie," "story" which appear across all five sentiment classes in roughly similar proportion, confirming these are structural/topic words rather than sentiment-bearing ones. As expected, these survived stopword filtering only because they're specific to movie review vocabulary, not because they carry polarity.

- "not" is the single most frequent word overall (~7,500 occurrences) heavily toward class 1 (somewhat negative) and class 3 (somewhat positive) rather than 
the extremes, consistent with negation typically softening or reversing a nearby sentiment word rather than marking a purely negative phrase on its own.

- "good", "funny" "love" are the clearest **sentiment-polarized word** in the top list. Its distribution is visibly weighted toward class 3/4 (positive) with a much smaller class 0/1 share than words like "film" or "movie," making it one of the few top-frequency words that's doing real sentiment work rather than just describing the domain.

- Class 2 (neutral) contributes a large, fairly uniform gray segment across almost every bar consistent with the earlier finding that neutral phrases dominate the dataset (~50% of rows)

**Takeaway:** raw top-word frequency is more useful for spotting domain vocabulary than sentiment signal because words like "good" stand out, but most of this list would benefit from a follow-up view (e.g. proportion within each class rather than raw stacked count) to surface words that are disproportionately common in one sentiment vs. spread evenly across all five.

In [ ]:
df_plot.head()

In [ ]:
sns.set_theme(style="whitegrid")

# pivot long format (Word, Sentiment, Count) into wide (Word x each Sentiment as a column)
wide = df_plot.pivot_table(index='Word', columns='Sentiment', values='Count', fill_value=0)
plot_df = wide.reset_index()

# ignore class 2 (neutral) entirely, as requested
plot_df['positive'] = plot_df.get(3, 0) + plot_df.get(4, 0)   # somewhat positive + positive
plot_df['negative'] = plot_df.get(0, 0) + plot_df.get(1, 0)   # negative + somewhat negative
plot_df['difference'] = plot_df['positive'] - plot_df['negative']
plot_df['leans_negative'] = plot_df['difference'] < 0

plot_df.head()

In [ ]:
# most polarized words, sorted so bars read cleanly top-to-bottom
top_words = (plot_df.reindex(plot_df['difference'].abs().sort_values(ascending=False).index)
             .head(25)
             .sort_values('difference'))
top_words.head()

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=top_words, y='Word', x='difference', hue='leans_negative',
            palette={True: '#26C6DA', False: '#F4837D'}, dodge=False)
plt.axvline(0, color='black', linewidth=0.8)
plt.legend(title='difference < 0')
plt.xlabel('difference (positive − negative occurrences)')
plt.ylabel('word')
plt.title('Words Most Associated with Positive vs Negative Sentiment')
plt.tight_layout()
plt.show()

#### Words Most Associated with Positive vs. Negative Sentiment:

Negation words dominate the negative side by a wide margin — "not" alone has a difference of roughly -1,800, more than double the next closest word ("no," ~-650). This is the clearest signal in the whole chart and directly validates preserving negation during stopword removal.

Genuine sentiment-bearing words anchor the positive side as expected: "good," "funny," "love," and "life" all show strong positive skew, consistent with straightforward positive adjectives/nouns showing up disproportionately in positive-labeled phrases.

Words like "way" and "time" sit essentially at zero, exactly where they should be — they're frequent across the corpus but carry no inherent polarity, so the chart correctly places them near the neutral line rather than pulling toward either side.

**Takeaway:** most of the top words behave exactly as expected (negation → negative, sentiment-adjectives → positive, generic nouns → neutral), which is a good sanity check that the cleaning pipeline preserved the right signal.

#### Target Analysis

In [ ]:
sns.countplot(data=train, x="Sentiment")

In [ ]:
print(train['Sentiment'].value_counts(normalize=True) * 100)

- The Neutral Sentiment class dominates the other class with a huge peak at ~8000. Which means it contributes around almost ~50% of the data. 
- The Positive and negative classes are ~1000 which is ~11% 
- The slightly positve and slightly negative are around ~3000 which is ~39%

## Pipeline Building

In [ ]:
print(train['SentenceId'].max(),test['SentenceId'].min())

**Data Leakage:**

Train data's max SentenceId is 8544, and Test data starts at SentenceId 8545. Kaggle didn't split this dataset by random row they split it by sentence, so every phrase from a given sentence stays entirely on one side. 

Because phrases from the same sentence share massive text overlap
- "A series of escapades demonstrating the adage" 
- "A series of escapades demonstrating"

differ by one word. If you split at the row/PhraseId level (like `train_test_split` normally does), siblings of the same sentence land on both sides of your split. The model doesn't generalize to that val phrase becuase it's already seen 90% of its words during training on its twin. The validation accuracy will look great and then it will not perform as well in testing.

**Solution:** 

`StratifiedGroupKFold (The Strategy)`
This combines two different cross-validation techniques into one:

`GroupKFold (groups=train['SentenceId'])`: It ensures that any phrases belonging to the same SentenceId stay entirely together in either the training set or the validation set. They are never split across both.

`StratifiedKFold (train['Sentiment'])`: It ensures that each of the 5 splits maintains roughly the same percentage of each sentiment class (0, 1, 2, 3, 4) as the original full dataset.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(train, train['Sentiment'], groups=train['SentenceId']))

train_split = train.iloc[train_idx].reset_index(drop=True)
val_split   = train.iloc[val_idx].reset_index(drop=True)

# mandatory sanity check — should print 0
overlap = set(train_split['SentenceId']) & set(val_split['SentenceId'])
print(f"Overlapping SentenceIds: {len(overlap)}")

In [ ]:
X_train = train_split['clean_phrase']
X_val = val_split['clean_phrase']
X_test = test['clean_phrase']

y_train = train_split['Sentiment']
y_val = val_split['Sentiment'] 

print(X_train.shape, X_val.shape, X_test.shape, "\n")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

def build_vectorizer(binary=False, tfidf=False, ngram_range=(1,2), stop_words=list(stop_words), min_df=3):
    if tfidf:
        return TfidfVectorizer(
            ngram_range=ngram_range, 
            stop_words=stop_words, 
            min_df=min_df
        )
    return CountVectorizer(
        ngram_range=ngram_range, 
        stop_words=stop_words, 
        min_df=min_df, 
        binary=binary
    )

In [ ]:
from sklearn.metrics import accuracy_score

class NBModelPipeline:
    def __init__(self, model, model_name, binary=False, tfidf=False, ngram_range=(1, 2), min_df=3):
        self.model_name = model_name
        self.binary = binary
        self.tfidf = tfidf
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.model_cls = model
        self.vectorizer = build_vectorizer(binary=binary, tfidf=tfidf, ngram_range=ngram_range, min_df=min_df)
        self.model = model()
        self.val_accuracy = None
        self.best_alpha = None
        self._X_train_vec = None
        self._X_val_vec = None

    def _vectorize(self, X_train, X_val):
        self._X_train_vec = self.vectorizer.fit_transform(X_train)
        self._X_val_vec = self.vectorizer.transform(X_val)

    def fit_and_evaluate(self, y_train, y_val, alpha=1.0):
        self.model.set_params(alpha=alpha)
        self.model.fit(self._X_train_vec, y_train)
        preds = self.model.predict(self._X_val_vec)
        self.val_accuracy = accuracy_score(y_val, preds)
        # print(f"[{self.model_name}] Alpha: {alpha} -> Val Accuracy: {self.val_accuracy:.4f}")
        return self.val_accuracy

    def alpha_hyper_param_tuning(self, X_train, y_train, X_val, y_val, coarse_alphas):
        self._vectorize(X_train, X_val)   # vectorize ONCE for this vectorizer config

        coarse_results = [(a, self.fit_and_evaluate(y_train, y_val, a)) for a in coarse_alphas]
        best_coarse_alpha = max(coarse_results, key=lambda x: x[1])[0]
        # print("\nBest coarse alpha:", best_coarse_alpha)

        fine_alphas = np.linspace(best_coarse_alpha / 5, best_coarse_alpha * 5, 15)
        fine_results = [(a, self.fit_and_evaluate(y_train, y_val, a)) for a in fine_alphas]

        self.best_alpha, self.val_accuracy = max(fine_results, key=lambda x: x[1])

        # leave self.model actually holding the winning alpha, not whichever ran last
        self.fit_and_evaluate(y_train, y_val, self.best_alpha)

        # print(f"\nBest fine-tuned alpha for {self.model_name}: {self.best_alpha:.4f} (val_accuracy={self.val_accuracy:.4f})")
        return self.best_alpha

    def full_grid_search(self, X_train, y_train, X_val, y_val, ngram_ranges, min_dfs, coarse_alphas):
        results = []

        for ngram_range in ngram_ranges:
            for min_df in min_dfs:
                # print(f"\n=== ngram_range={ngram_range}, min_df={min_df} ===")
                self.ngram_range, self.ngram_range = ngram_range, min_df
                self.vectorizer = build_vectorizer(
                    binary=self.binary, tfidf=self.tfidf,
                    ngram_range=ngram_range, min_df=min_df,
                )

                best_alpha = self.alpha_hyper_param_tuning(X_train, y_train, X_val, y_val, coarse_alphas)

                results.append({
                    'ngram_range': ngram_range,
                    'min_df': min_df,
                    'best_alpha': best_alpha,
                    'val_accuracy': self.val_accuracy,
                    'vocab_size': len(self.vectorizer.vocabulary_),
                })

        results_df = pd.DataFrame(results).sort_values('val_accuracy', ascending=False)
        best_result = results_df.loc[results_df['val_accuracy'].idxmax()]
        print(best_result)
        self.best_alpha, self.val_accuracy = best_result['best_alpha'], best_result['val_accuracy']
        self.ngram_range, self.ngram_range = best_result['ngram_range'], best_result['ngram_range']
        # print("\nTop configs:\n", results_df.head())
        return results_df

In [ ]:
hyper_params = {
    'ngram_range': [(1, 1), (1, 2), (1, 3)],
    'min_df': [2, 3, 5],
    'alphas': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0],
}

In [ ]:
from sklearn.naive_bayes import BernoulliNB

bnb = NBModelPipeline(BernoulliNB, "BernoulliNB",binary= True, tfidf= False)

bnb.full_grid_search(X_train, y_train, X_val, y_val, hyper_params['ngram_range'], hyper_params['min_df'], hyper_params['alphas'])

In [ ]:
from sklearn.naive_bayes import MultinomialNB

mnb = NBModelPipeline(MultinomialNB, "Multinomial NB",binary= False, tfidf= False)

mnb.full_grid_search(X_train, y_train, X_val, y_val, hyper_params['ngram_range'], hyper_params['min_df'], hyper_params['alphas'])

In [ ]:
from sklearn.naive_bayes import MultinomialNB
tnb = NBModelPipeline(MultinomialNB, "MNB TF-IDF ",binary= False, tfidf= True)

tnb.full_grid_search(X_train, y_train, X_val, y_val, hyper_params['ngram_range'], hyper_params['min_df'], hyper_params['alphas'])

In [ ]:
from sklearn.naive_bayes import ComplementNB

cnb = NBModelPipeline(ComplementNB, "Complement NB",binary= False,tfidf=True)

cnb.full_grid_search(X_train, y_train, X_val, y_val, hyper_params['ngram_range'], hyper_params['min_df'], hyper_params['alphas'])

### Comparing Models 

In [ ]:
rows = []
for pipeline in [bnb, mnb, tnb, cnb]:
    preds = pipeline.model.predict(pipeline._X_val_vec)
    report = classification_report(y_val, preds, output_dict=True)
    rows.append({
        'model': pipeline.model_name,
        'accuracy': report['accuracy'],
        'macro_f1': report['macro avg']['f1-score'],
        'recall_negative': report['0']['recall'],
        'recall_positive': report['4']['recall'],
    })

comparison_df = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
print(comparison_df)

In [ ]:
metrics_df = comparison_df.set_index('model')

# scale each metric column to 0-1 independently — this is what makes
# "highest in THIS metric" always map to green, even though accuracy
# and recall_negative live on totally different scales
normalized = metrics_df.apply(lambda col: (col - col.min()) / (col.max() - col.min()), axis=0)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(normalized, annot=metrics_df, fmt='.3f', cmap='RdYlGn',
            cbar=False, linewidths=1, linecolor='white', ax=ax)
plt.title('Model Comparison Across Metrics\n(green = best per metric, red = worst per metric)')
plt.ylabel('')
plt.tight_layout()
plt.show()

- Multinomial Naive Bayes clearnly outperforms the other NB models in all metrics except for recall_negative. Since the model is built on count of words and the dataset has cropped reviews and this makes it more menaingful or more confirming of the sentiemnt of each word to the model. 
- Bernoulli's NB performs the worst of all 4 models. It's probably becase the model focuses on the presence of the word in the reviews and since the dataset is built on by cropping other reviews this model doesn't perform well.  

In [ ]:
labels = ["Negative", "Slightly Neg", "Neutral", "Slightly Pos", "Positive"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

mnb_preds = mnb.model.predict(mnb._X_val_vec)
cm_mnb = confusion_matrix(y_val, mnb_preds)

sns.heatmap(cm_mnb, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0], cbar=False)
axes[0].set_title(f"{mnb.model_name} Confusion Matrix", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Predicted Labels", fontsize=11)
axes[0].set_ylabel("True Labels", fontsize=11)

cnb_preds = cnb.model.predict(cnb._X_val_vec)
cm_cnb = confusion_matrix(y_val, cnb_preds)

sns.heatmap(cm_cnb, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title(f"{cnb.model_name} Confusion Matrix", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Predicted Labels", fontsize=11)
axes[1].set_ylabel("True Labels", fontsize=11)

plt.tight_layout()
plt.show()

**Model Comparison**

Multinomial NB (raw counts) wins outright this time — best accuracy, best macro-F1, and 
even best recall on the positive class. With the earlier duplicate-prediction bug fixed, 
there's no longer a tie for second place obscuring the picture.

**Notably, TF-IDF does *not* outperform raw counts here** — the opposite of what we found on 
the IMDB project. Likely explanation: TF-IDF's main strength is discounting words that are 
common across *many* documents, which matters when documents are long enough for word 
repetition and document overlap to be meaningful signals. At this dataset's typical 5-7 word 
phrase length, that reweighting has much less to work with, and may even remove useful signal 
rather than add it. Worth remembering as a general lesson: TF-IDF isn't a strict upgrade over 
raw counts — its benefit is dataset-dependent, not automatic.

**Complement NB shows a genuine, informative tradeoff, not just a worse model.** It scores by 
far the best recall on the negative class (0.151 vs. 0.112) — its bias-correction mechanism is 
doing exactly what it's built to do for that one class. But it pays for this with a large 
accuracy drop (0.475, ~9 points below the winner) and doesn't carry the same benefit over to 
recall_positive, where it's actually slightly behind plain Multinomial NB. The net effect is a 
lower macro-F1 despite the recall win — meaning the correction is likely overcorrecting 
elsewhere (probably misclassifying a meaningful chunk of Neutral phrases into the extremes), 
costing more than the negative-class gain is worth once all 5 classes are weighed equally. 
This is a different story than the SMS project, where ComplementNB's tradeoff was simple 
(precision vs. recall on one class) — with 5 ordinal classes, the same mechanism has more 
places to go wrong.

BernoulliNB remains the weakest on macro-F1 and recall_negative, consistent with the earlier 
explanation: presence/absence scoring is a weak signal when most of the vocabulary is 
necessarily "absent" from any single short phrase.

**Takeaway:** Multinomial NB (raw counts) is the clear choice for final model selection — it's 
not just the highest accuracy, it's the best macro-F1 too, meaning the win isn't coming purely 
from dominating the Neutral majority class.

#### Final Model and Test

In [ ]:
sample.head()

In [ ]:
test.head()

In [ ]:
print(X_train.shape , X_val.shape , y_train.shape, y_val.shape)

X_train_full = pd.concat([X_train, X_val]).reset_index(drop=True)
y_train_full = pd.concat([y_train, y_val]).reset_index(drop=True)
X_test = test['clean_phrase']

print(f'New X Train : {X_train_full.shape} , y train : {y_train_full.shape}, X test : {X_test.shape}')

print(f'Best - alpha : {mnb.best_alpha}, ngram_range : {mnb.ngram_range}, min_df: {mnb.min_df}')

In [ ]:
vectorizer = build_vectorizer(binary=False, tfidf=False, ngram_range=mnb.ngram_range, min_df=mnb.min_df)
X_train_t = vectorizer.fit_transform(X_train_full)
X_test_t = vectorizer.transform(X_test)

model = MultinomialNB(alpha=mnb.best_alpha)
model.fit(X_train_t,y_train_full)

In [ ]:
preds = model.predict(X_test_t)
preds.shape

In [ ]:
submission_df = pd.DataFrame({
    'PhraseId': test['PhraseId'],
    'Sentiment': preds
})

print(submission_df.shape)
print(submission_df.head())

In [ ]:
submission_df.to_csv('submission.csv', index=False)